# Chapter 4: Baker's Method Setup

**Context:**
This script establishes the constants and bounds required to apply Baker's Method to the Thue equation $F_t(X,Y) = \mu$.

**Methodology:**
1. **Root Distances & Beta Bounds (Lemma 4.3):** Evaluates the minimal distances between the roots of $f_t$ and establishes upper bounds on $|\beta^{(j)}| = |x - \alpha^{(j)}y|$ for Lemma 4.1.2, which is subsequently used to obtain an upper bound on our linear form in two logarithms.
2. **Cramer's Rule Bounds (Lemma 4.5):** Solves the system of equations defined by the logarithms of the independent units using Cramer's rule. This bounds the unknown integer coefficients $v_1$ and $v_2$ in terms of $\log|y|$, used in the application of Laurent's and Mignotte's theorems in subsequent steps.

**Key Functions:**
* `verify_beta_bounds()`: Computes the minimum root differences and bounds for Lemma 4.1.2.
* `verify_cramer_bounds()`: Calculates the maximal $y$-coefficients and constant terms for $|v_1/I|$ and $|v_2/I|$ (Lemma 4.1.4).

In [ ]:
import itertools
from sage.all import *

# =============================================================================
# GLOBAL SETTINGS
# =============================================================================
RUN_BAKER_BOUNDS_SEARCH = True 

RR_val = RealField(200)
CC_val = ComplexField(200)

# =============================================================================
# SHARED HELPER FUNCTIONS
# =============================================================================

def generate_imaginary_d_list(M):
    """Returns a list of squarefree positive integers d for bounds up to M."""
    dlist = []
    for d in range(1, floor(4 * M**2 - 1) + 1):
        rem = d % 4
        if rem in (1, 2) and d <= M**2 and Integer(d).is_squarefree():
            dlist.append(d)
        elif rem == 3 and Integer(d).is_squarefree():
            dlist.append(d)
    return dlist

def get_imaginary_quad_ints(M):
    """Returns a dictionary of valid imaginary quadratic integers up to bound M."""
    dlist = generate_imaginary_d_list(M)
    big_set_of_integers = {}
    x_qq = PolynomialRing(QQ, 'x').gen()
    
    for d in dlist:
        ablist = set()
        L = NumberField(x_qq**2 + d, 'rootd')
        rootd = L.gen()
        for i in range(0, floor(M**2) + 1):
            elts = L.elements_of_norm(i)
            if Integer(i).is_square():
                iroot = L(sqrt(i))
                for u in L.roots_of_unity():
                    elts.append(iroot * u)
            for j in elts:
                a = QQ(j.trace() / 2)
                b = QQ((j - a) / rootd)
                ablist.add((sgn(a) * a, b))
        big_set_of_integers[d] = ablist
        
    return dlist, big_set_of_integers

def generate_valid_t_roots_cc(m_bound):
    """
    Generator that yields the absolute value of t and the roots.
    """
    dlist, all_integers = get_imaginary_quad_ints(m_bound)
    
    x_qq = PolynomialRing(QQ, 'x').gen()
    CC_poly = PolynomialRing(CC_val, 'x')
    x_cc = CC_poly.gen()
    
    for d in dlist:
        for a, b in all_integers[d]:
            if b == 0:  # Skip rational integers
                continue
                
            abst = float(sqrt(a**2 + d * b**2))
            
            # Check irreducibility over Number Field L_t
            dpol = x_qq**2 - 2 * a * x_qq + a**2 + d * b**2
            if Integer((2 * a)**2 - 4 * (a**2 + d * b**2)).is_square():
                continue
                
            L = NumberField(dpol, 't')
            t_nf = L.gen()
            R = PolynomialRing(L, 'y')
            y = R.gen()
            ft = y**4 - t_nf * y**3 - 6 * y**2 + t_nf * y + 1
            
            if not ft.is_irreducible():
                continue
                
            # Complex numerical evaluation to bypass symbolic sorting errors
            tt = CC_val(a + sqrt(-d)*b)
            ftt = x_cc**4 - tt * x_cc**3 - 6 * x_cc**2 + tt * x_cc + 1
            ftp = ftt.derivative()
            
            # Extract numerical roots and identify alpha0
            cc_roots = [r[0] for r in ftt.roots()]
            alpha0 = min(cc_roots, key=abs)
            
            # Generate the remaining roots from root relations
            alpha1 = (alpha0 - 1) / (alpha0 + 1)
            alpha2 = -1 / alpha0
            alpha3 = -(alpha0 + 1) / (alpha0 - 1)
            
            roots = [alpha0, alpha1, alpha2, alpha3]
            t_string = f"{a} + {b}*sqrt({-d})"
            
            yield t_string, abst, roots, ftp


# =============================================================================
# PART 1: Beta Bounds and Minimal Distances (Lemma 4.1.2)
# =============================================================================

def apply_sigma(k, j):
    """Applies the automorphism sigma^(j) to a root k."""
    if j == 0:   return k
    elif j == 1: return (k - 1) / (k + 1)
    elif j == 2: return -1 / k
    elif j == 3: return -(k + 1) / (k - 1)

def verify_beta_bounds(m_bound=10):
    """
    Finds the minimum of |f'_t(alpha_j)| / |t| and the minimum distance 
    between alpha_j and any other root for |t| < m_bound.
    """
    overall_min_denom = {0: 10.0, 1: 10.0, 2: 10.0, 3: 10.0}
    overall_min_diff = 2.0
    
    print("Generating field extensions and scanning Beta bounds...")
    
    for t_str, abst, roots, ftp in generate_valid_t_roots_cc(m_bound):
        # Min distance between any two roots
        min_diff = min(abs(r1 - r2) for r1, r2 in itertools.combinations(roots, 2))
        if min_diff < overall_min_diff:
            overall_min_diff = min_diff
            
        alpha0 = roots[0]
        
        for j in range(4):
            alphaj = apply_sigma(alpha0, j)
            bddenom = float(abs(ftp(alphaj)) / abst)
            if bddenom < overall_min_denom[j]:
                overall_min_denom[j] = bddenom
                
    return overall_min_denom, float(overall_min_diff)


# =============================================================================
# PART 2: Cramer's Rule Bounds (Lemma 4.1.4)
# =============================================================================

def calculate_cramer_bounds(roots, j):
    """
    Calculates the bounds on v1/I and v2/I for a given set of roots.
    Returns (v1_y_coeff, v1_cst, v2_y_coeff, v2_cst)
    """
    alpha0 = roots[0]
    logalpha = float(log(abs(alpha0)))
    logalphaprime = float(log(abs(roots[1])))
    denom = sum(float(log(abs(r)))**2 for r in roots)
    
    thesis_cst = 2 * 132.22 / (20**4)
    
    if j == 0:
        logalphaplus1 = float(log(abs(alpha0 + 1)))
        logalpha2plus1 = float(log(abs(alpha0**2 + 1)))
        
        c1_cst = abs(2 * logalpha2plus1 - logalpha - logalphaplus1) + thesis_cst
        c2_cst = abs(-logalphaprime) + thesis_cst
        
        v1_y_coef = abs(-4 * logalpha)
        v1_cstterm = abs(-2 * logalpha * c1_cst) + abs(c2_cst * (logalpha + logalphaprime))
        
        v2_y_coef = abs(-4 * logalphaprime)
        v2_cstterm = abs(-2 * logalphaprime * c1_cst) + abs(c2_cst * (-logalpha + logalphaprime))
        
    elif j == 3:
        a0_prime = roots[1]
        logalphaprimeplus1 = float(log(abs(a0_prime + 1)))
        logalphaprime2plus1 = float(log(abs(a0_prime**2 + 1)))
        
        c1_cst = abs(2 * logalphaprime2plus1 - logalphaprime - logalphaprimeplus1) + thesis_cst
        c2_cst = abs(-logalpha) + thesis_cst
        
        v1_y_coef = abs(4 * logalphaprime)
        v1_cstterm = abs(-2 * logalphaprime * c1_cst) + abs(c2_cst * (-logalpha + logalphaprime))
        
        v2_y_coef = abs(-4 * logalpha)
        v2_cstterm = abs(-2 * logalpha * c1_cst) + abs(c2_cst * (logalpha + logalphaprime))
        
    else:
        raise ValueError("Only j=0 and j=3 are supported for Cramer bounds.")
        
    return (float(v1_y_coef/denom), float(v1_cstterm/denom), 
            float(v2_y_coef/denom), float(v2_cstterm/denom))

def verify_cramer_bounds(m_bound=10):
    """
    Finds the maximum values of the coefficients r_i and s_i such that 
    |v_i/I| < r_i*log|y| + s_i for j=0 and j=3.
    """
    max_bounds = {
        0: {'v1_y': 0.0, 'v1_c': 0.0, 'v2_y': 0.0, 'v2_c': 0.0},
        3: {'v1_y': 0.0, 'v1_c': 0.0, 'v2_y': 0.0, 'v2_c': 0.0}
    }
    
    print("Generating field extensions and scanning Cramer bounds...")
    
    for t_str, abst, roots, ftp in generate_valid_t_roots_cc(m_bound):
        for j in [0, 3]:
            v1_y, v1_c, v2_y, v2_c = calculate_cramer_bounds(roots, j)
            
            if v1_y > max_bounds[j]['v1_y']: max_bounds[j]['v1_y'] = v1_y
            if v1_c > max_bounds[j]['v1_c']: max_bounds[j]['v1_c'] = v1_c
            if v2_y > max_bounds[j]['v2_y']: max_bounds[j]['v2_y'] = v2_y
            if v2_c > max_bounds[j]['v2_c']: max_bounds[j]['v2_c'] = v2_c
            
    return max_bounds


# =============================================================================
# MAIN RUNNER
# =============================================================================

if __name__ == "__main__":
    if RUN_BAKER_BOUNDS_SEARCH:
        print("=== Lemma 4.3: Beta Bounds & Minimal Distances ===")
        min_denoms, overall_min_diff = verify_beta_bounds(m_bound=10)
        
        print(f"  • Overall min difference between roots: {overall_min_diff:.4f}")
        for j in range(4):
            print(f"  • Overall min |f'_t(alpha_{j})| / |t| : {min_denoms[j]:.4f}")
            
        print("\n=== Lemma 4.5: Cramer's Rule Bounds ===")
        max_cramer = verify_cramer_bounds(m_bound=10)
        
        for j in [0, 3]:
            print(f"  • For j = {j}:")
            print(f"      |v1/I| < {max_cramer[j]['v1_y']:.2f}*log|y| + {max_cramer[j]['v1_c']:.2f}")
            print(f"      |v2/I| < {max_cramer[j]['v2_y']:.2f}*log|y| + {max_cramer[j]['v2_c']:.2f}")
    else:
        print("Search disabled. Set RUN_BAKER_BOUNDS_SEARCH = True to execute.")

=== Lemma 4.3: Beta Bounds & Minimal Distances ===
Generating field extensions and scanning Beta bounds...
  • Overall min difference between roots: 0.6764
  • Overall min |f'_t(alpha_0)| / |t| : 0.1865
  • Overall min |f'_t(alpha_1)| / |t| : 0.3031
  • Overall min |f'_t(alpha_2)| / |t| : 0.8078
  • Overall min |f'_t(alpha_3)| / |t| : 0.3031

=== Lemma 4.5: Cramer's Rule Bounds ===
Generating field extensions and scanning Cramer bounds...
  • For j = 0:
      |v1/I| < 2.73*log|y| + 1.24
      |v2/I| < 1.77*log|y| + 1.48
  • For j = 3:
      |v1/I| < 1.77*log|y| + 1.98
      |v2/I| < 2.73*log|y| + 1.90
